In [ ]:
# Keep imports and data/output paths relative to the repository root.
import os
import sys
from pathlib import Path
from datetime import datetime
from uuid import uuid4

for repository_dir in (Path.cwd(), *Path.cwd().parents):
    if (repository_dir / ".geofm-root").is_file():
        os.chdir(repository_dir)
        break
else:
    raise FileNotFoundError("Open this notebook from inside the repository.")
if "src" not in sys.path:
    sys.path.insert(0, "src")


### **Imports**

In [ ]:
import torch
import torch.nn as nn
import rasterio
import numpy as np
import matplotlib.pyplot as plt

from use_croma import PretrainedCROMA

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

pretrained_path = "pretrained/croma/CROMA_large.pt"
path_to_the_dataset = "datasets/sen1floods11/sen1floods11_hand_labeled_only/HandLabeled/"

path_to_X = path_to_the_dataset + "S1Hand/"

path_to_Y = path_to_the_dataset + "LabelHand/"

print(device)

cuda


### **Functions**

### **Experiment**

In [3]:
with rasterio.open(path_to_X + "Ghana_103272_S1Hand.tif") as src:
    image = src.read()
    print("Image shape:", image.shape)
    print("Image dtype:", image.dtype)
    print("Band descriptions:", src.descriptions)
    print("Min/Max:", np.nanmin(image), np.nanmax(image))
    print("Mean:", np.nanmean(image, axis=(1, 2)))
    print("Std:", np.nanstd(image, axis=(1, 2)))
    print("Nodata:", src.nodata)

with rasterio.open(path_to_Y + "Ghana_103272_LabelHand.tif") as src:
    mask = src.read(1)
    print("Mask shape:", mask.shape)
    print("Label values/counts:", np.unique(mask, return_counts=True))

Image shape: (2, 512, 512)
Image dtype: float32
Band descriptions: ('VV', 'VH')
Min/Max: -47.37703 8.5869255
Mean: [ -9.208067 -15.214209]
Std: [2.2619739 2.5052812]
Nodata: nan
Mask shape: (512, 512)
Label values/counts: (array([-1,  0,  1], dtype=int16), array([  2875, 257711,   1558]))


In [5]:
print("NaN per band:", np.isnan(image).sum(axis=(1, 2)))
print("Inf per band:", np.isinf(image).sum(axis=(1, 2)))

print(
    "Ignored pixels:",
    (mask == -1).sum()
)

print(
    "Water percentage:",
    100 * (mask == 1).sum() / (mask != -1).sum()
)

NaN per band: [0 0]
Inf per band: [0 0]
Ignored pixels: 2875
Water percentage: 0.6009202797094909


In [6]:
def normalize(x, use_8_bit):
    # taken from CROMA that took it from SatMAE and SeCo
    x = x.float()

    imgs = []
    for channel in range(x.shape[1]):
        min_value = x[:, channel, :, :].mean() - 2 * x[:, channel, :, :].std()
        max_value = x[:, channel, :, :].mean() + 2 * x[:, channel, :, :].std()

        if use_8_bit:
            img = (x[:, channel, :, :] - min_value) / (max_value - min_value) * 255.0
            img = torch.clip(img, 0, 255).unsqueeze(dim=1).to(torch.uint8)
            imgs.append(img)
        else:
            img = (x[:, channel, :, :] - min_value) / (max_value - min_value)
            img = torch.clip(img, 0, 1).unsqueeze(dim=1)
            imgs.append(img)

    return torch.cat(imgs, dim=1)

In [7]:
if not isinstance(image, torch.Tensor):
    image = torch.from_numpy(image)
    image = image.unsqueeze(0)

image = normalize(image, False)

print(image)

print(image.shape)
print(image.dtype)
print(image.min(), image.max())

tensor([[[[0.4166, 0.4318, 0.5054,  ..., 0.5712, 0.2770, 0.2697],
          [0.0469, 0.1741, 0.3152,  ..., 0.6242, 0.4767, 0.3607],
          [0.2528, 0.2765, 0.2955,  ..., 0.5341, 0.5541, 0.5759],
          ...,
          [0.4987, 0.6508, 0.6162,  ..., 0.5891, 0.6494, 0.6976],
          [0.6894, 0.7376, 0.5456,  ..., 0.4507, 0.4382, 0.4524],
          [0.8062, 0.7265, 0.4216,  ..., 0.4142, 0.4505, 0.1020]],

         [[0.8360, 0.7327, 0.6334,  ..., 0.2744, 0.3663, 0.7210],
          [0.7586, 0.6473, 0.6502,  ..., 0.0165, 0.4177, 0.7355],
          [0.5530, 0.3847, 0.4654,  ..., 0.5313, 0.6710, 0.7835],
          ...,
          [0.6012, 0.4990, 0.5280,  ..., 0.5565, 0.7533, 0.5943],
          [0.5625, 0.5088, 0.5398,  ..., 0.4090, 0.5432, 0.4171],
          [0.1570, 0.3872, 0.4614,  ..., 0.3243, 0.2366, 0.1824]]]])
torch.Size([1, 2, 512, 512])
torch.float32
tensor(0.) tensor(1.)


In [8]:
if not isinstance(mask, torch.Tensor):
    mask = torch.from_numpy(mask)
    mask = mask.unsqueeze(0)
    
    
print(mask.shape)
print(mask.dtype)
print(mask.min(), mask.max())

torch.Size([1, 512, 512])
torch.int16
tensor(-1, dtype=torch.int16) tensor(1, dtype=torch.int16)


In [9]:
crop_size = 120

# Choose random crop coordinates
top = torch.randint(0, 512 - crop_size + 1, (1,)).item()
left = torch.randint(0, 512 - crop_size + 1, (1,)).item()

# Crop both tensors at the same location
image_120_crop = image[..., top:top+crop_size, left:left+crop_size]
mask_120_crop = mask[..., top:top+crop_size, left:left+crop_size]

print(image_120_crop.shape)
print(mask_120_crop.shape)

print("Water pixels:", (mask_120_crop == 1).sum().item())
print("Non-water pixels:", (mask_120_crop == 0).sum().item())
print("Ignored pixels:", (mask_120_crop == -1).sum().item())

torch.Size([1, 2, 120, 120])
torch.Size([1, 120, 120])
Water pixels: 0
Non-water pixels: 14400
Ignored pixels: 0


#### Loading the pretrained encoder

In [ ]:
croma = PretrainedCROMA(pretrained_path= pretrained_path, 
                        size='large', 
                        modality='SAR', 
                        image_resolution=120)

croma = croma.to(device)

Initializing SAR encoder


In [11]:
croma.requires_grad_(False)
croma.eval()

PretrainedCROMA(
  (s1_encoder): ViT(
    (linear_input): Linear(in_features=128, out_features=1024, bias=True)
    (transformer): BaseTransformer(
      (layers): ModuleList(
        (0-11): 12 x ModuleList(
          (0): Attention(
            (to_qkv): Linear(in_features=1024, out_features=3072, bias=False)
            (to_out): Linear(in_features=1024, out_features=1024, bias=True)
            (input_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (1): FFN(
            (net): Sequential(
              (0): Linear(in_features=1024, out_features=4096, bias=True)
              (1): GELU(approximate='none')
              (2): Dropout(p=0.0, inplace=False)
              (3): Linear(in_features=4096, out_features=1024, bias=True)
            )
            (input_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
          )
        )
      )
      (norm_out): LayerNo

#### Testing the input/output interface

In [12]:
image_120_crop = image_120_crop.to(device)

In [13]:
with torch.no_grad():
    outputs = croma(SAR_images=image_120_crop)

features = outputs["SAR_encodings"]

features.shape

torch.Size([1, 225, 1024])

#### Preparing the embeddings for the FCN head

In [14]:
features = outputs["SAR_encodings"]
B, N, D = features.shape
H = W = int(N ** 0.5)

spatial_features = features.transpose(1, 2).reshape(B, D, H, W)

spatial_features.shape

torch.Size([1, 1024, 15, 15])

#### Attaching the FCN head

In [15]:
from torchvision.models.segmentation.fcn import FCNHead

head = FCNHead(in_channels=1024, channels=2)

head[1] = nn.GroupNorm(
    num_groups=16,
    num_channels=256
)

head = head.to(device)

print(head)

logits = head(spatial_features)

logits.shape

FCNHead(
  (0): Conv2d(1024, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (1): GroupNorm(16, 256, eps=1e-05, affine=True, bias=True)
  (2): ReLU()
  (3): Dropout(p=0.1, inplace=False)
  (4): Conv2d(256, 2, kernel_size=(1, 1), stride=(1, 1))
)


torch.Size([1, 2, 15, 15])

In [16]:
import torch.nn as nn



#### Upsampling

In [17]:
import torch.nn.functional as F

In [18]:
pixel_logits = F.interpolate(
    logits,
    size=(120, 120),
    mode="bilinear",
    align_corners=False
)

print(pixel_logits.shape)

torch.Size([1, 2, 120, 120])


#### Verifying the model can compute a segmentation loss

In [19]:
mask_120_crop = mask_120_crop.long().to(device)

In [20]:
criterion = torch.nn.CrossEntropyLoss(ignore_index=-1)

loss = criterion(pixel_logits, mask_120_crop)
print(loss.item())

1.1292973756790161


#### Veryfying which parameters will be trained

In [21]:
def count_trainable(model):
    return sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )

print("CROMA:", count_trainable(croma))
print("FCNHead:", count_trainable(head))

CROMA: 0
FCNHead: 2360322


In [25]:
optimizer = torch.optim.AdamW(head.parameters(), lr=1e-4)

head.train()

for step in range(20):
    logits = head(spatial_features)

    pixel_logits = F.interpolate(
        logits,
        size=mask_120_crop.shape[-2:],
        mode="bilinear",
        align_corners=False
    )

    loss = criterion(pixel_logits, mask_120_crop)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Step {step+1}: Loss = {loss.item():.8f}")

Step 1: Loss = 0.00000198
Step 2: Loss = 0.00000153
Step 3: Loss = 0.00000104
Step 4: Loss = 0.00000135
Step 5: Loss = 0.00000107
Step 6: Loss = 0.00000106
Step 7: Loss = 0.00000103
Step 8: Loss = 0.00000083
Step 9: Loss = 0.00000085
Step 10: Loss = 0.00000078
Step 11: Loss = 0.00000098
Step 12: Loss = 0.00000084
Step 13: Loss = 0.00000086
Step 14: Loss = 0.00000063
Step 15: Loss = 0.00000071
Step 16: Loss = 0.00000058
Step 17: Loss = 0.00000070
Step 18: Loss = 0.00000060
Step 19: Loss = 0.00000054
Step 20: Loss = 0.00000058


In [23]:
head.eval()

with torch.no_grad():
    logits = head(spatial_features)
    logits = F.interpolate(
        logits,
        size=mask_120_crop.shape[-2:],
        mode="bilinear",
        align_corners=False
    )

    predictions = logits.argmax(dim=1)

print("Predicted classes:", torch.unique(predictions, return_counts=True))
print("Actual classes:", torch.unique(mask_120_crop, return_counts=True))

Predicted classes: (tensor([0], device='cuda:0'), tensor([14400], device='cuda:0'))
Actual classes: (tensor([0], device='cuda:0'), tensor([14400], device='cuda:0'))


In [24]:
for mode in ["train", "eval"]:
    head.train(mode == "train")

    with torch.no_grad():
        logits = head(spatial_features)
        logits = F.interpolate(
            logits,
            size=mask_120_crop.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        predictions = logits.argmax(dim=1)
        loss = criterion(logits, mask_120_crop)

    print(mode)
    print("Loss:", loss.item())
    print("Predictions:", torch.unique(predictions, return_counts=True))

train
Loss: 1.692337036729441e-06
Predictions: (tensor([0], device='cuda:0'), tensor([14400], device='cuda:0'))
eval
Loss: 1.5228464462779812e-06
Predictions: (tensor([0], device='cuda:0'), tensor([14400], device='cuda:0'))
